# 🎙️ LeadMaster CRM - CosyVoice 2 무료 GPU 클라우드 음성 복제 서버

이 노트북은 **Google Colab의 무료 T4 GPU(16GB VRAM)**를 활용하여,
우리 리드마스터 CRM의 실제 통화 녹취 기반 **10인 연령대별 가상 고객(20대~70대)**의 목소리를 실시간으로 복제하여 말하게 하는 **FastAPI + Cloudflare Tunnel 음성 서버**입니다.

### ⚡ 사용 방법 (간단 3단계):
1. 상단 메뉴에서 `런타임` -> `런타임 유형 변경` -> **T4 GPU** 선택 (기본 선택됨)
2. 아래의 각 셀을 순서대로 실행 (단축키: `Shift + Enter`)
3. 마지막 셀에 나오는 **`https://xxxx.trycloudflare.com`** 주소를 복사하여 **리드마스터 CRM 설정**에 입력하면 끝!

In [ ]:
# 1. GPU 사양 확인 (Tesla T4 16GB가 확인되어야 합니다)
!nvidia-smi

In [ ]:
# 2. CosyVoice 2 공식 오픈소스 리포지토리 클론 및 종속성 설치
!git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git
%cd CosyVoice
!pip install -q -r requirements.txt
!pip install -q fastapi uvicorn pyngrok pydantic torchaudio soundfile librosa

# CosyVoice 2 사전학습 모델 다운로드 (ModelScope)
from modelscope import snapshot_download
snapshot_download('iic/CosyVoice2-0.5B', local_dir='pretrained_models/CosyVoice2-0.5B')

In [ ]:
# 3. 10인 레퍼런스 음원 및 서버 코드 다운로드 (또는 업로드된 zip 압축 해제)
# 작업 폴더에서 colab_cosyvoice_pack.zip을 Colab 왼쪽 파일 탭으로 드래그해 넣으시면 자동으로 풀립니다.
import os, zipfile

if os.path.exists('/content/colab_cosyvoice_pack.zip'):
    with zipfile.ZipFile('/content/colab_cosyvoice_pack.zip', 'r') as z:
        z.extractall('/content/CosyVoice')
    print('✅ 10인 보이스 팩 압축 해제 완료!')
else:
    print('⚠️ /content/colab_cosyvoice_pack.zip 파일이 없습니다. 왼쪽 파일 탭에 zip 파일을 업로드해 주세요.')

In [ ]:
# 4. Cloudflare Tunnel 무료 도메인 도구 다운로드
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# 5. 백엔드 FastAPI 서버 백그라운드 실행 & Cloudflare Tunnel 공용 URL 발급!
import subprocess, time, re

# FastAPI 서버 백그라운드 구동
server_proc = subprocess.Popen(['python', 'server.py'])
time.sleep(3)

# Cloudflare Tunnel 구동
tunnel_proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'], stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True)

public_url = None
print('🔗 Cloudflare Tunnel 연결 중...')
for _ in range(30):
    line = tunnel_proc.stderr.readline()
    if 'trycloudflare.com' in line:
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0)
            break
    time.sleep(1)

if public_url:
    print('\n' + '='*60)
    print('🎉 [성공] 리드마스터 CRM 전용 CosyVoice 음성 서버가 시작되었습니다!')
    print(f'👉 CRM에 복사할 주소: {public_url}')
    print('='*60 + '\n')
    print('이 주소를 CRM 트레이닝 화면 설정창의 [CosyVoice Colab 서버 URL]에 붙여넣으세요.')
else:
    print('❌ 터널 연결 지연. tunnel_proc.stderr 로그를 확인해 주세요.')